# Export Imagery and Covariate Values for Asian Countries

In [ ]:
import ee
import math
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
from shapely.geometry import Point
from osgeo import gdal
from eeutils import SpatialCovariates, Sentinel2SR, LandsatSR, get_utm_epsg, extract_patches, extract_spatial_covariates, get_neighbourhood, plot_neighbourhood

In [ ]:
# Imagery Export Settings **CHANGE THESE**
COUNTRIES = ["BGD", "IND", "IDN", "KGZ", "KHM", "MMR", "NPL", "PAK", "PHL", "TLS", "SLB", "COK", "FJI"]
COMPOSITE_START_YEAR = 2010                        # start year to image composite
COMPOSITE_END_YEAR = 2010                          # end year for image composite
IMAGERY_TYPES = ['AEF', 'COVS']                    # imagery to export; options = ['AEF', 'COVS', 'LANDSAT', 'SENTINEL']  

In [ ]:
# Paths
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth/"
ADM0_DIR = HOME_DIR / "data-raw/boundaries/ADM0/"

COUNTRY_PATCHES_OUTDIR = HOME_DIR / "data-raw/imagery/country/patches"          # full-resolution S2 imagery
COG_OUTDIR = HOME_DIR / "data-raw/imagery/country/cogs/"                        # mosaicked per-country COGs
GRID_OUTDIR = HOME_DIR / "data-clean/outputs/maps/map-grids/"

# Export Params
PROJECT = 'historical-infrastructure'              # GCloud project attached to Earth Engine
N_CONCURRENT_REQUESTS = 5                          # Earth Engine default limit is 40
OVERWRITE_EXISTING_PATCHES = False                 # skip patch if same-named patch exists


In [ ]:
# Satellite Image Bands
SENTINEL_BANDS = ("B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12")      # bands for Sentinel imagery
LANDSAT_BANDS = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']              # bands to select

# Spatial Covariate Params
COV_NODATA_VAL = 0                   # if only masked pixels, replace pixel w. zero
COV_LANDCOVER_RES = 100              # downsample DynamicWorld to 100m before aggregating
COV_MAX_PIXELS = 1000                # max number of input pixels to create an output pixel
COV_BEST_EFFORT = True               # use higher-level input layers if computation involves more than MAX_PIXELS

# AlphaEarth Foundation Embedding Params
AEF_NODATA_VAL = -128                 # matches nodata value of Google's raw quantised AEF asset

# Sampling Grid Params
SCALE = {'LANDSAT': 30, 'SENTINEL': 10, 'COVS': 500, 'AEF': 500} 
PATCH_SIZE = 256                     # extract a 256 x 256 pixel patch
ADM0_BUFFER_FRACTION = 1.5           # buffer for ADM0 country boundary, as multiple of patch resolution

# Nodata value written into each imagery type's COG mosaic (None = no explicit nodata band)
NODATA_VALUES = {'LANDSAT': None, 'SENTINEL': None, 'COVS': COV_NODATA_VAL, 'AEF': AEF_NODATA_VAL}

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT, opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
def quantise_aef(image: ee.Image, nodata_val: int = AEF_NODATA_VAL) -> ee.Image:
    """
    Re-quantise AlphaEarth Foundations embeddings back to signed int8.

    Earth Engine automatically de-quantises the AEF asset to float32. This inverts that de-quantisation 
    (see https://developers.google.com/earth-engine/guides/aef_on_gcs_readme) before export.
    """
    quantised = image.abs().sqrt().multiply(127.5).multiply(image.signum()).round().clamp(-127, 127)
    return quantised.unmask(nodata_val).toInt8()

In [ ]:
# Landsat
landsat_image = LandsatSR(start_date=f"{COMPOSITE_START_YEAR}-01-01", end_date=f"{COMPOSITE_END_YEAR}-12-31", bands=LANDSAT_BANDS, 
                            rescale_bands=False, mask_clouds=True).images.median().uint16()

# Sentinel
sentinel_image = Sentinel2SR(start_date=f"{COMPOSITE_START_YEAR}-01-01", end_date=f"{COMPOSITE_END_YEAR}-12-31", 
                             keep_qa=False, rescale=False, bands=SENTINEL_BANDS).images.median().uint16()

# Spatial covariates
covs_image = SpatialCovariates(year=COMPOSITE_START_YEAR, landcover_scale=COV_LANDCOVER_RES, nodata_val=COV_NODATA_VAL, 
                  max_pixels=COV_MAX_PIXELS, best_effort=COV_BEST_EFFORT).all_covariates_comp

# AlphaEarth Foundation Embeddings (re-quantised to int8 to minimise export size)
aef_image = quantise_aef(
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f'{COMPOSITE_START_YEAR}-01-01', f'{COMPOSITE_END_YEAR}-12-31')
    .mosaic()
)

IMAGES = {'LANDSAT': landsat_image, 'SENTINEL': sentinel_image, 'COVS': covs_image, 'AEF': aef_image}


In [ ]:
# Check that image to sample from defined correctly
manila = (120.9842, 14.5995)
manila_crs = get_utm_epsg(manila)
example = get_neighbourhood(image=landsat_image, pt=manila, patch_scale=30, patch_crs=manila_crs)
plot_neighbourhood(patches=example, levels=2, bands=["RED", "GREEN", "BLUE"], vis_min=7500, vis_max=16000)

### Export Image Patches and Matching Covariate Statistics

In [ ]:
# --- BUILD GRIDS AT REQUESTED RESOLUTION

export_df = []
for imagery_type in IMAGERY_TYPES:
    
    STRIDE = PATCH_SIZE * SCALE[imagery_type]                       # stride in m (for UTM)
    
    for country in COUNTRIES:
    
        # Load country boundary; get UTM CRS based on country centroid
        adm0 = gpd.read_file(ADM0_DIR / f"{country.lower()}_adm0.geojson").dissolve()
        assert len(adm0) == 1
        pt = adm0.to_crs(epsg=6933).centroid.to_crs(epsg=4326).iloc[0]
        export_crs = get_utm_epsg((pt.x, pt.y))
    
        # Reproject country boundary to UTM and buffer
        adm0_buffer = STRIDE * ADM0_BUFFER_FRACTION
        adm0 = adm0.to_crs(export_crs).buffer(adm0_buffer)
        adm0_polygon = adm0[0]
    
        # Get bounds for ADM0 and snap to multiple of Sentinel stride
        xmin, ymin, xmax, ymax = adm0_polygon.bounds
        grid_xmin = math.floor(xmin / STRIDE) * STRIDE
        grid_ymin = math.floor(ymin / STRIDE) * STRIDE
        grid_xmax = math.ceil(xmax / STRIDE) * STRIDE
        grid_ymax = math.ceil(ymax / STRIDE) * STRIDE
    
        # Create grid of patch centroids over full snapped bounds
        x_centroids = np.arange(grid_xmin + STRIDE/2, grid_xmax, STRIDE)
        y_centroids = np.arange(grid_ymax - STRIDE/2, grid_ymin, -STRIDE)
    
        # Save as GeoDataFrame
        grid_pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x in x_centroids for y in y_centroids], crs=export_crs)
        grid_pts["patch_id"] = (f"{country.upper()}_{imagery_type}_{COMPOSITE_START_YEAR}0101{COMPOSITE_END_YEAR}1231_" +
                              "EPSG" + str(grid_pts.crs.to_epsg()) +
                              grid_pts.geometry.x.astype(int).map(lambda v: f"{'M' if v < 0 else 'P'}{abs(v):08d}") +
                              grid_pts.geometry.y.astype(int).map(lambda v: f"{'M' if v < 0 else 'P'}{abs(v):08d}")
                              )
    
        grid_pts['country'] = country.upper()
        grid_pts['imagery_type'] = imagery_type.upper()
        grid_pts['grid_stride'] = STRIDE
        grid_pts['grid_x'] = np.rint((((grid_pts.geometry.x - STRIDE/2) - grid_xmin) / STRIDE)).astype(int)        # x, y pixel coordinates in grid
        grid_pts["grid_y"] = np.rint((grid_ymax - (grid_pts.geometry.y + STRIDE/2)) / STRIDE).astype(int)          # ...
        grid_pts['utm_x'] = grid_pts.geometry.x
        grid_pts['utm_y'] = grid_pts.geometry.y
        grid_cells = grid_pts.geometry.buffer(STRIDE/2, cap_style=3)
        grid_pts['intersects_adm0'] = grid_cells.intersects(adm0_polygon)
        grid_pts['epsg_crs'] = export_crs
        grid_adm0 = grid_pts.loc[grid_pts["intersects_adm0"]].drop(columns="intersects_adm0")
    
        # Sanity check -- confirm sampling grid falls within ADM0
        print(f"Building Grid for {country.upper()}-{imagery_type.upper()}")
        ax = adm0.plot(facecolor="none", edgecolor="black", linewidth=2)
        grid_adm0.geometry.plot(ax=ax, color="red", markersize=0.02)
        plt.show()
    
        # Drop coordinates and concatenate
        export_df.append(grid_adm0.drop(columns=grid_adm0.geometry.name))
    
        # Save grid
        GRID_OUTDIR.mkdir(parents=True, exist_ok=True)
        grid_adm0.to_parquet(GRID_OUTDIR / f"{country.lower()}_{imagery_type.lower()}_grid_pts_adm0.parquet")
        grid_pts.to_parquet(GRID_OUTDIR / f"{country.lower()}_{imagery_type.lower()}_grid_pts.parquet")
    
export_df = pd.concat(export_df)


In [ ]:
# --- Export Image Patches

for (country, imagery_type), country_df in export_df.groupby(['country', 'imagery_type']):

    print(f"Exporting {len(country_df)} patches for {country}-{imagery_type}:")
    country_crs = country_df.epsg_crs.iloc[0]
    country_pts = [(x, y) for x, y in zip(country_df.utm_x, country_df.utm_y)]
    country_ids = list(country_df['patch_id'])

    imagery_export_dir = COUNTRY_PATCHES_OUTDIR / imagery_type.upper() / country.upper()
    imagery_export_dir.mkdir(parents=True, exist_ok=True)
    extract_patches(
        image=IMAGES[imagery_type],
        pts=country_pts,
        pts_crs=country_crs,
        patch_ids=country_ids,
        patch_size=PATCH_SIZE,
        patch_scale=SCALE[imagery_type],
        patch_crs=country_crs,
        output_dir=imagery_export_dir,
        overwrite_patches=OVERWRITE_EXISTING_PATCHES,
        concurrent_requests=N_CONCURRENT_REQUESTS
    )

### Build Cloud-Optimized GeoTIFF Mosaics

In [ ]:
def _predictor_for_dtype(gdal_dtype: int) -> int:
    """DEFLATE predictor: 3 (floating point) for float bands, 2 (horizontal differencing) otherwise."""
    return 3 if gdal_dtype in (gdal.GDT_Float32, gdal.GDT_Float64) else 2


def patches_to_cog(patch_dir, out_path, nodata=None, blocksize=256, compress="DEFLATE",
                    num_threads="ALL_CPUS", cache_gb=8):
    """Mosaic a directory of patch GeoTIFFs into a single Cloud-Optimized GeoTIFF."""
    tif_paths = sorted(str(p) for p in Path(patch_dir).glob("*.tif"))
    if not tif_paths:
        raise FileNotFoundError(f"No .tif patches found in {patch_dir}")

    out_path = Path(out_path)
    vrt_path = out_path.with_suffix(".vrt")

    with gdal.config_options({"GDAL_CACHEMAX": str(cache_gb * 1024), "GDAL_NUM_THREADS": str(num_threads)}):
        vrt_kwargs = {"srcNodata": nodata, "VRTNodata": nodata} if nodata is not None else {}
        vrt_ds = gdal.BuildVRT(str(vrt_path), tif_paths, **vrt_kwargs)
        predictor = _predictor_for_dtype(vrt_ds.GetRasterBand(1).DataType)
        vrt_ds = None  # flush VRT to disk before translating

        gdal.Translate(
            str(out_path), str(vrt_path),
            format="COG",
            creationOptions=[
                f"BLOCKSIZE={blocksize}",
                f"COMPRESS={compress}",
                f"PREDICTOR={predictor}",
                "BIGTIFF=IF_SAFER",
                f"NUM_THREADS={num_threads}",
                "OVERVIEWS=NONE",
                "OVERVIEW_RESAMPLING=NEAREST",
            ],
            noData=nodata,
        )

    vrt_path.unlink(missing_ok=True)

In [ ]:
# --- Build one COG mosaic per country/imagery-type

for imagery_type in IMAGERY_TYPES:
    for country in COUNTRIES:
        patch_dir = COUNTRY_PATCHES_OUTDIR / imagery_type / country
        if not patch_dir.exists():
            continue

        COG_OUTDIR.mkdir(parents=True, exist_ok=True)
        out_path = COG_OUTDIR / f"{country.lower()}_{imagery_type.lower()}_s{COMPOSITE_START_YEAR}_e{COMPOSITE_END_YEAR}.tif"

        print(f"Building COG for {country}-{imagery_type} -> {out_path}")
        patches_to_cog(patch_dir, out_path, nodata=NODATA_VALUES[imagery_type])